# Microsoft Fabric — Custom Library (.whl) End-to-End Guide

> **Purpose:** This notebook is a complete teaching reference for packaging a custom Python library,  
> uploading it to a Microsoft Fabric Environment, and using it inside a Fabric Notebook.
>
> **Author:** Myla Ram Reddy  
> **Platform:** Microsoft Fabric (Spark Runtime 1.3 — Spark 3.5, Delta 3.2)  
> **Reference:** [Microsoft Learn — Manage Libraries in Fabric Environment](https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library)

---

## What You Will Learn

| Step | What happens | Where |
|------|-------------|-------|
| 1 | Understand Fabric Environments | Concept |
| 2 | Create the Python package folder structure | Local PC |
| 3 | Write the library code (`dq_utils`) | Local PC |
| 4 | Build the `.whl` file | Local PC — Terminal |
| 5 | Upload `.whl` to Fabric Environment | Fabric Browser UI |
| 6 | Publish the Environment | Fabric Browser UI |
| 7 | Attach Environment to a Notebook | Fabric Notebook |
| 8 | Import and use the custom library | Fabric Notebook |
| 9 | Run full Data Quality checks | Fabric Notebook |

---

## SECTION 1 — What is a Microsoft Fabric Environment?

A **Fabric Environment** is a reusable configuration capsule that contains:

- **Spark Runtime** — which version of Apache Spark to use (e.g. Runtime 1.3 = Spark 3.5)
- **Libraries** — public PyPI/Conda packages AND your own custom `.whl` / `.jar` / `.tar.gz` files
- **Spark Compute** — node size, executor memory, number of cores
- **Spark Properties** — custom Spark configuration settings

You attach one Environment to many Notebooks and Spark Job Definitions.  
Everyone who runs those items gets **exactly the same libraries and settings** — no manual installs needed.

### Why Custom Libraries?

Instead of copy-pasting the same utility functions across 20 notebooks, you:

1. Write the functions **once** in a Python package
2. Package them into a `.whl` file
3. Upload to Fabric Environment
4. Every notebook that attaches the environment can `import` your functions immediately

### Supported Custom Library File Types

| File Type | Language | Notes |
|-----------|----------|-------|
| `.whl` | Python | Recommended for Python packages |
| `.py` | Python | Single script file |
| `.jar` | Java / Scala | Not supported in Quick mode |
| `.tar.gz` | R | R language packages |

> **Official Docs:** https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library

---

## SECTION 2 — Create the Project Folder Structure (Local PC)

Everything in this section runs on your **local Windows / Mac / Linux machine** — NOT inside Fabric.

Open **Command Prompt** (Windows) or **Terminal** (Mac/Linux) and run:

```bash
# Navigate to your Desktop (or any folder you prefer)
cd C:\Users\YourName\Desktop

# Create the outer project folder
mkdir dq_utils
cd dq_utils

# Create the inner Python package folder (same name)
mkdir dq_utils
```

Your folder structure should look like this:

```
dq_utils/                  ← outer project folder
│
├── dq_utils/              ← inner Python package folder
│   ├── __init__.py        ← tells Python this is a package
│   └── checks.py          ← your actual library functions
│
└── setup.py               ← build configuration
```

> **Note:** The outer `dq_utils/` is your project workspace.  
> The inner `dq_utils/` is the actual Python package that gets imported.

---

## SECTION 3 — Write the Library Code (Local PC)

Create the following **3 files** inside your project folder using VS Code, Notepad, or any editor.

---

### File 1 of 3 — `dq_utils/checks.py`

This file contains all the reusable Data Quality functions.

Copy the code from **Cell [3a]** below.

---

### File 2 of 3 — `dq_utils/__init__.py`

This file exposes your functions when someone does `from dq_utils import ...`.

Copy the code from **Cell [3b]** below.

---

### File 3 of 3 — `setup.py` (in the outer folder)

This file tells Python how to build and package your library.

Copy the code from **Cell [3c]** below.

In [ ]:
# =============================================================
# FILE: dq_utils/checks.py
# Save this as: C:\Users\YourName\Desktop\dq_utils\dq_utils\checks.py
# =============================================================

from pyspark.sql.functions import col, sum as spark_sum, isnan, when


def null_count(df):
    """
    Returns a dictionary of {column_name: null_count} for a PySpark DataFrame.
    Counts both None and NaN values.

    Args:
        df: PySpark DataFrame

    Returns:
        dict: {column_name: int}

    Example:
        null_count(df)  →  {'id': 0, 'name': 1, 'age': 2}
    """
    result = {}
    for c in df.columns:
        null_n = df.select(
            spark_sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0))
        ).collect()[0][0]
        result[c] = null_n
    return result


def duplicate_count(df, subset=None):
    """
    Returns the number of duplicate rows in a PySpark DataFrame.

    Args:
        df: PySpark DataFrame
        subset: list of column names to check duplicates on (default: all columns)

    Returns:
        int: number of duplicate rows

    Example:
        duplicate_count(df)                        →  2
        duplicate_count(df, subset=['name','age']) →  1
    """
    cols = subset if subset else df.columns
    total = df.count()
    distinct = df.select(cols).distinct().count()
    return total - distinct


def row_count(df):
    """
    Returns the total number of rows in a PySpark DataFrame.

    Args:
        df: PySpark DataFrame

    Returns:
        int: total row count
    """
    return df.count()


def schema_summary(df):
    """
    Prints column names, data types, and nullable flags.

    Args:
        df: PySpark DataFrame
    """
    print(f"{'Column':<20} {'DataType':<15} {'Nullable'}")
    print("-" * 45)
    for field in df.schema.fields:
        print(f"{field.name:<20} {str(field.dataType):<15} {field.nullable}")


def run_all_checks(df, subset_cols=None):
    """
    Runs all Data Quality checks and prints a full summary report.

    Checks performed:
        - Total row count
        - Duplicate row count
        - Null / NaN count per column
        - Schema summary (column names + data types)

    Args:
        df: PySpark DataFrame
        subset_cols: list of columns to check for duplicates (default: all)

    Example:
        run_all_checks(df)
        run_all_checks(df, subset_cols=['name', 'age'])
    """
    print("=" * 45)
    print("       DATA QUALITY REPORT — dq_utils")
    print("=" * 45)

    print(f"\n  Total rows     : {row_count(df)}")
    print(f"  Duplicate rows : {duplicate_count(df, subset_cols)}")

    print("\n  Null counts per column:")
    for col_name, count in null_count(df).items():
        status = "OK" if count == 0 else "!!"
        print(f"    [{status}]  {col_name}: {count} nulls")

    print("\n  Schema:")
    for field in df.schema.fields:
        print(f"    {field.name:<20} {str(field.dataType)}")

    print("\n" + "=" * 45)

In [ ]:
# =============================================================
# FILE: dq_utils/__init__.py
# Save this as: C:\Users\YourName\Desktop\dq_utils\dq_utils\__init__.py
# =============================================================

from .checks import null_count, duplicate_count, row_count, schema_summary, run_all_checks

__version__ = "1.0.0"
__author__  = "Your Name"
__all__     = ["null_count", "duplicate_count", "row_count", "schema_summary", "run_all_checks"]

In [ ]:
# =============================================================
# FILE: setup.py
# Save this as: C:\Users\YourName\Desktop\dq_utils\setup.py
# NOTE: This is in the OUTER dq_utils folder, NOT the inner one
# =============================================================

from setuptools import setup, find_packages

setup(
    name="dq_utils",
    version="1.0.0",
    packages=find_packages(),
    description="Custom Data Quality Utility Library for Microsoft Fabric",
    author="Your Name",
    python_requires=">=3.8",
)

## SECTION 4 — Build the `.whl` File (Local PC Terminal)

Open your terminal, navigate to the **outer** `dq_utils` folder, and run these commands:

```bash
# Step 1: Navigate to your project folder
cd C:\Users\YourName\Desktop\dq_utils

# Step 2: Install build tools (only needed once)
pip install wheel setuptools

# Step 3: Build the wheel file
python setup.py bdist_wheel
```

### What the Terminal Output Means

```
running bdist_wheel              ← starts building the wheel
running build                    ← prepares source files
running build_py
creating build/lib/dq_utils
copying dq_utils/__init__.py     ← copies your files into build folder
copying dq_utils/checks.py
SetuptoolsDeprecationWarning     ← WARNING ONLY — safe to ignore
creating 'dist/dq_utils-1.0.0-py3-none-any.whl'  ← YOUR FILE IS CREATED
adding 'dq_utils/__init__.py'
adding 'dq_utils/checks.py'
removing build/bdist...          ← cleans up temp files
```

> ⚠️ The `SetuptoolsDeprecationWarning` is **NOT an error**. Your build succeeds even with this warning.

### After the Build — Your File Location

```
dq_utils/
└── dist/
    └── dq_utils-1.0.0-py3-none-any.whl   ← THIS IS YOUR FILE (≈ 2 KB)
```

Open **File Explorer** → Desktop → dq_utils → dist → you will see the `.whl` file.

---

## SECTION 5 — Upload `.whl` to Fabric Environment (Browser UI)

Everything from here onwards happens inside **Microsoft Fabric in your browser**.

### Steps

1. Open your browser → go to **https://app.fabric.microsoft.com**
2. Click your **Workspace** from the left nav
3. Click **New item** → select **Environment**
4. Give it a name (e.g. `my_env`) → click **Create**
5. In the left nav of the Environment editor → click **Libraries** → **Custom**
6. Click **Upload** in the toolbar
7. Browse to `Desktop → dq_utils → dist → dq_utils-1.0.0-py3-none-any.whl`
8. Select it and click **Open**
9. You will see the file appear with **Status: Saved** and **Last updated: New**

### What You Should See After Upload

```
Custom libraries
─────────────────────────────────────────────────────
Language                          Status    Last updated
dq_utils-1.0.0-py3-none-any.whl  Saved     New
─────────────────────────────────────────────────────
```

> **Note:** Make sure you are on the **Full mode** tab (not Quick mode).  
> `.whl` files work in both modes, but `.jar` files only work in Full mode.

---

## SECTION 6 — Publish the Environment (Browser UI)

Uploading alone is NOT enough. You must **Publish** for the library to be installed on the Spark cluster.

### Steps

1. In the Environment editor, click the **Home** tab (top left)
2. Click **Publish** button in the toolbar
3. A **Pending changes** review screen appears — check your library is listed
4. Click **Publish all**
5. A blue banner appears: *"Your changes are currently being published"*
6. Click **View progress** to watch the publish stages live

### Publish Timeline

```
Validating...          (1-2 min)
Installing libraries   (3-10 min)   ← installs your .whl on all Spark nodes
Environment ready  ✅
```

> **Total time:** 5 to 15 minutes depending on the number and size of libraries.  
> Leave the browser tab open. You can move on to create the Notebook while waiting.

---

## SECTION 7 — Attach the Environment to a Notebook (Browser UI)

### Steps

1. In your Workspace → click **New item** → select **Notebook**
2. The notebook opens with one empty cell
3. In the **top toolbar**, find the **Environment dropdown**
   - It shows `Missing environment` or `Workspace default` by default
4. Click the dropdown → select **my_env**
5. The toolbar now shows `my_env` — you are connected

### Toolbar Reference

```
[ Run all ]  [ Stop ]  [ Environment: my_env ▾ ]  [ + Code ]  [ + Markdown ]
                                ↑
                        Click here and select my_env
```

> **Important:** If the Environment dropdown is greyed out, your Spark session is running.  
> Click **Stop session** first, then change the environment, then run your cells.

---

## SECTION 8 — Verify the Library is Installed (Fabric Notebook)

Run the cell below. This confirms `dq_utils` is available in your Spark environment.

In [ ]:
# Cell 1 — Verify your custom library is installed
# Expected output: dq_utils    1.0.0

%pip list | grep dq_utils

### Expected Output
```
dq_utils          1.0.0
Note: you may need to restart the kernel to use updated packages.
```

> The note about restarting the kernel is just a reminder — if you just published the environment,  
> you do NOT need to restart. If you updated an existing library, restart your Spark session.

Now import the library:

In [ ]:
# Cell 2 — Import all functions from dq_utils

from dq_utils import null_count, duplicate_count, row_count, schema_summary, run_all_checks

print("dq_utils imported successfully!")
print(f"Available functions: null_count, duplicate_count, row_count, schema_summary, run_all_checks")

---

## SECTION 9 — Use the Library: Full Data Quality Walkthrough (Fabric Notebook)

Now we put the library to real use. We create a realistic dataset with intentional data quality issues  
and run each function to detect them.

### The Dataset — Employee Records with Known Issues

| id | name    | age  | department  | Issue |
|----|---------|------|-------------|-------|
| 1  | Alice   | 29   | Sales       | None |
| 2  | Bob     | None | Engineering | **Null age** |
| 3  | Charlie | 35   | Sales       | None |
| 4  | Alice   | 29   | Sales       | **Duplicate of row 1** |
| 5  | None    | 40   | HR          | **Null name** |
| 6  | Diana   | 31   | None        | **Null department** |

In [ ]:
# Cell 3 — Create a PySpark DataFrame with intentional data quality issues

data = [
    (1, "Alice",   29,   "Sales"),        # clean row
    (2, "Bob",     None, "Engineering"),  # null age
    (3, "Charlie", 35,   "Sales"),        # clean row
    (4, "Alice",   29,   "Sales"),        # duplicate of row 1
    (5, None,      40,   "HR"),           # null name
    (6, "Diana",   31,   None),           # null department
]

df = spark.createDataFrame(data, ["id", "name", "age", "department"])

print("DataFrame created. Showing all rows:")
df.show(truncate=False)

In [ ]:
# Cell 4 — Function 1: row_count()
# Returns the total number of rows

total = row_count(df)
print(f"Total rows in DataFrame: {total}")
# Expected output: 6

In [ ]:
# Cell 5 — Function 2: null_count()
# Returns a dictionary of {column: null_count}

nulls = null_count(df)
print("Null counts per column:")
for column, count in nulls.items():
    flag = "CLEAN" if count == 0 else f"HAS {count} NULL(s)"
    print(f"  {column:<15} → {flag}")

# Expected output:
# id              → CLEAN
# name            → HAS 1 NULL(s)
# age             → HAS 1 NULL(s)
# department      → HAS 1 NULL(s)

In [ ]:
# Cell 6 — Function 3: duplicate_count()
# Check duplicates across ALL columns

dupes_all = duplicate_count(df)
print(f"Duplicate rows (all columns): {dupes_all}")

# Check duplicates across SPECIFIC columns only
dupes_subset = duplicate_count(df, subset=["name", "age", "department"])
print(f"Duplicate rows (name+age+dept): {dupes_subset}")

# Expected output:
# Duplicate rows (all columns): 1
# Duplicate rows (name+age+dept): 1

In [ ]:
# Cell 7 — Function 4: schema_summary()
# Prints column names, data types, and nullable flags

schema_summary(df)

# Expected output:
# Column               DataType        Nullable
# ---------------------------------------------
# id                   LongType()      True
# name                 StringType()    True
# age                  LongType()      True
# department           StringType()    True

In [ ]:
# Cell 8 — Function 5: run_all_checks()
# Runs ALL checks in one call and prints a full DQ report

run_all_checks(df)

# Expected output:
# =============================================
#        DATA QUALITY REPORT — dq_utils
# =============================================
#   Total rows     : 6
#   Duplicate rows : 1
#
#   Null counts per column:
#     [OK]  id: 0 nulls
#     [!!]  name: 1 nulls
#     [!!]  age: 1 nulls
#     [!!]  department: 1 nulls
#
#   Schema:
#     id                   LongType()
#     name                 StringType()
#     age                  LongType()
#     department           StringType()
# =============================================

---

## SECTION 10 — Real World Usage Pattern

In production, you would run DQ checks on data loaded from your **Lakehouse** — not from a hardcoded list.

Here is the pattern your students should use when reading real data:

In [ ]:
# Cell 9 — Real world pattern: Read from Lakehouse and run DQ checks
# NOTE: Replace the path below with your actual Lakehouse table name

from dq_utils import run_all_checks

# Option A: Read from a Lakehouse Delta table
# df = spark.read.format("delta").load("Tables/your_table_name")

# Option B: Read from a Lakehouse CSV file in Files section
# df = spark.read.option("header", True).csv("Files/your_file.csv")

# Option C: Read using SparkSQL
# df = spark.sql("SELECT * FROM your_lakehouse.your_table")

# Once you have your DataFrame, run DQ checks in ONE line:
# run_all_checks(df)

print("Uncomment one of the options above and replace with your table/file name.")
print("Then call run_all_checks(df) to get your full Data Quality Report.")

---

## SECTION 11 — Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| `ModuleNotFoundError: No module named 'dq_utils'` | Environment not attached | Check toolbar — select `my_env` from the Environment dropdown |
| `ModuleNotFoundError: No module named 'dq_utils'` | Publish not finished | Wait for publish to complete (5–15 min), then **Stop session** and re-run |
| `%pip list` shows nothing for dq_utils | Session started before publish | Click **Stop session** → restart → run cells again |
| Environment dropdown is greyed out | Spark session is active | Click **Stop session** first, then change the environment |
| Upload button is greyed out | Publish is in progress | Wait for current publish to finish before uploading again |
| `SetuptoolsDeprecationWarning` during build | Old-style setup.py | Safe to ignore — build still succeeds |

---

## SECTION 12 — Key Reference Links

| Resource | URL |
|----------|-----|
| Fabric Environment — create and use | https://learn.microsoft.com/en-us/fabric/data-engineering/create-and-use-environment |
| Manage libraries in Fabric Environment | https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library |
| Fabric Spark runtimes | https://learn.microsoft.com/en-us/fabric/data-engineering/runtime |
| Fabric library management best practices | https://learn.microsoft.com/en-us/fabric/data-engineering/library-management |
| Fabric Environment public REST API | https://learn.microsoft.com/en-us/fabric/data-engineering/environment-public-api |
| Fabric data engineering overview | https://learn.microsoft.com/en-us/fabric/data-engineering/data-engineering-overview |

---

## Summary

```
LOCAL PC                          MICROSOFT FABRIC (Browser)
─────────────────────────         ──────────────────────────────────
1. Write checks.py                5. Upload .whl to Environment
2. Write __init__.py              6. Publish Environment (5-15 min)
3. Write setup.py                 7. Attach Environment to Notebook
4. Run: python setup.py           8. from dq_utils import run_all_checks
        bdist_wheel               9. run_all_checks(df)  → DQ Report
   → dist/dq_utils-1.0.0.whl
```

> **Key insight for students:** The `.whl` file is just a zip file containing your Python code  
> with metadata. Once uploaded to a Fabric Environment and published, it behaves exactly  
> like any PyPI package — you import it the same way you import `pandas` or `numpy`.

---
*Notebook created by Myla Ram Reddy | Microsoft Fabric Custom Library Teaching Reference*